## DynamoDB CRUD Operations

Welcome to the next step in your AWS development journey! So far, you have learned the boto3 Universal Pattern for connecting to AWS services and how to work with S3 for cloud file storage. In this lesson, we will focus on DynamoDB, Amazon's managed NoSQL database service.

DynamoDB is designed for fast and flexible data storage and retrieval. It is often used for applications that need to handle large amounts of data with low latency, such as user profiles, session data, or product catalogs. By the end of this lesson, you will know how to create a DynamoDB table and perform the four basic operations: Create, Read, Update, and Delete (CRUD).

This lesson will build on your experience with the boto3 library in Python but will focus only on what you need to get started with DynamoDB. Let's get started!

---

## Recall: Using boto3 to Access AWS Services

Before we dive into DynamoDB, let's quickly remind ourselves how we use boto3 to connect to AWS services in Python. You have already used boto3 to work with S3, and the process is very similar for DynamoDB.

To use boto3, you typically import the library and create a resource or client for the AWS service you want to use. For example, to work with S3, you wrote:

```python
import boto3

s3 = boto3.resource('s3')
```

For DynamoDB, you will use:

```python
import boto3

dynamodb = boto3.resource('dynamodb')
```

This line creates a DynamoDB resource object, which you will use to interact with your tables. On CodeSignal, boto3 is already installed, so you do not need to worry about setup here, but remember how to do this for your own environment.

---

## Creating a DynamoDB Table in Python

Let's start by creating a DynamoDB table. In DynamoDB, a table is where your data is stored. Each table needs a name and a primary key. The primary key uniquely identifies each item in the table.

Here's how you can create a table named `Users_<random>` with a primary key called `user_id`:

```python
import uuid
import boto3

dynamodb = boto3.resource('dynamodb')
TABLE_NAME = f"Users_{uuid.uuid4().hex[:8]}"

table = dynamodb.create_table(
    TableName=TABLE_NAME,
    KeySchema=[{"AttributeName": "user_id", "KeyType": "HASH"}],
    AttributeDefinitions=[{"AttributeName": "user_id", "AttributeType": "S"}],
    BillingMode='PAY_PER_REQUEST'
)
```

Let's break this down:

* `TableName=TABLE_NAME` sets the name of the table. Here, we use a random suffix to avoid name conflicts.
* `KeySchema` defines the primary key. In this case, `user_id` is the only key, and it is a `"HASH"` key (the partition key).
* `AttributeDefinitions` tells DynamoDB that `user_id` is a string (`"S"`).
* `BillingMode='PAY_PER_REQUEST'` means you pay only for what you use, which is good for learning and small projects.

After creating the table, you need to wait until it is ready to use. Here's a helper function to do that:

```python
import time

def wait_active(name):
    client = boto3.client('dynamodb')
    for _ in range(30):
        if client.describe_table(TableName=name)["Table"]["TableStatus"] == "ACTIVE":
            return
        time.sleep(2)
    raise TimeoutError("Table not active in time")
```

You call this function after creating the table:

```python
wait_active(TABLE_NAME)
print("Table created:", TABLE_NAME)
```

**Sample Output:**

```text
Table created: Users_1a2b3c4d
```

---

## Performing CRUD Operations on DynamoDB

Now that you have a table, let's go through the four basic operations: Create, Read, Update, and Delete.

### Create: Adding a New Item

To add a new item (a row) to your table, use the `put_item` method:

```python
table.put_item(Item={"user_id": "u-123", "name": "Ada", "email": "ada@example.com"})
```

This adds a user with ID `u-123`, name `Ada`, and email `ada@example.com` to the table.

### Read: Retrieving an Item

To get an item by its primary key, use `get_item`:

```python
response = table.get_item(Key={"user_id": "u-123"})
print("Read:", response.get("Item"))
```

**Sample Output:**

```text
Read: {'user_id': 'u-123', 'name': 'Ada', 'email': 'ada@example.com'}
```

### Update: Modifying an Existing Item

To update an item, use `update_item`. For example, to change the email address:

```python
updated = table.update_item(
    Key={"user_id": "u-123"},
    UpdateExpression="SET email = :e",
    ExpressionAttributeValues={":e": "ada.lovelace@example.com"},
    ReturnValues="ALL_NEW"
)
print("Updated:", updated["Attributes"])
```

* `UpdateExpression` tells DynamoDB what to change.
* `ExpressionAttributeValues` provides the new value.
* `ReturnValues="ALL_NEW"` returns the updated item.

**Sample Output:**

```text
Updated: {'user_id': 'u-123', 'name': 'Ada', 'email': 'ada.lovelace@example.com'}
```

### Delete: Removing an Item

To delete an item, use `delete_item`:

```python
table.delete_item(Key={"user_id": "u-123"})
print("Deleted user u-123")
```

**Sample Output:**

```text
Deleted user u-123
```

---

## Cleaning Up: Deleting the Table

It's important to clean up resources you no longer need. To delete the table:

```python
table.delete()
print("Dropped table:", TABLE_NAME)
```

**Sample Output:**

```text
Dropped table: Users_1a2b3c4d
```

Deleting unused tables helps you avoid unnecessary costs and keeps your AWS account tidy.

---

## Summary and Practice Preview

In this lesson, you learned how to create a DynamoDB table and perform the four basic CRUD operations using Python and boto3:

* Creating a table with a primary key
* Adding, reading, updating, and deleting items
* Cleaning up by deleting the table

You saw step-by-step code examples and learned what each part does. In the next practice exercises, you will get hands-on experience with these operations. Remember to pay attention to table names and primary keys, and always clean up your resources when you are done. Good luck, and enjoy working with DynamoDB!

## Fixing DynamoDB Table Creation Timing

Now that you understand how to create DynamoDB tables and perform basic operations, let's tackle a common issue that developers face when working with AWS services.

You have been given code that creates a DynamoDB table and immediately tries to add an item to it. However, there's a timing problem — DynamoDB table creation is asynchronous, which means the table needs time to become active before you can use it.

Your task is to fix this by adding the missing `wait_active()` function call in the correct location. Look for the TODO comment that shows you exactly where to add this call. The `wait_active()` function is already provided for you.

Once you add the missing line, your code will successfully create the table, wait for it to become ready, add a user, read the user back, and clean up properly. This exercise will teach you an important lesson about working with AWS resources and proper timing in cloud applications.

```python
import time, uuid, boto3
from botocore.exceptions import ClientError

dynamodb = boto3.resource('dynamodb')
client = boto3.client('dynamodb')
TABLE_NAME = f"Users_{uuid.uuid4().hex[:8]}"

def wait_active(name):
    for _ in range(30):
        if client.describe_table(TableName=name)["Table"]["TableStatus"] == "ACTIVE":
            return
        time.sleep(2)
    raise TimeoutError("Table not active in time")

def main():
    # Create table
    table = dynamodb.create_table(
        TableName=TABLE_NAME,
        KeySchema=[{"AttributeName": "user_id", "KeyType": "HASH"}],
        AttributeDefinitions=[{"AttributeName": "user_id", "AttributeType": "S"}],
        BillingMode='PAY_PER_REQUEST'
    )
    # TODO: Add the wait_active function call here to wait for the table to be ready
    print("Table created:", TABLE_NAME)

    # Create item
    table.put_item(Item={"user_id": "u-123", "name": "Ada", "email": "ada@example.com"})
    
    # Read item
    response = table.get_item(Key={"user_id": "u-123"})
    print("Read:", response.get("Item"))

    # Cleanup
    table.delete()
    print("Dropped table:", TABLE_NAME)

if __name__ == "__main__":
    main()
```

Here is the completed code with the `wait_active()` call added right after the table is created:

```python
import time, uuid, boto3
from botocore.exceptions import ClientError

dynamodb = boto3.resource('dynamodb')
client = boto3.client('dynamodb')
TABLE_NAME = f"Users_{uuid.uuid4().hex[:8]}"

def wait_active(name):
    for _ in range(30):
        if client.describe_table(TableName=name)["Table"]["TableStatus"] == "ACTIVE":
            return
        time.sleep(2)
    raise TimeoutError("Table not active in time")

def main():
    # Create table
    table = dynamodb.create_table(
        TableName=TABLE_NAME,
        KeySchema=[{"AttributeName": "user_id", "KeyType": "HASH"}],
        AttributeDefinitions=[{"AttributeName": "user_id", "AttributeType": "S"}],
        BillingMode='PAY_PER_REQUEST'
    )
    wait_active(TABLE_NAME)
    print("Table created:", TABLE_NAME)

    # Create item
    table.put_item(Item={"user_id": "u-123", "name": "Ada", "email": "ada@example.com"})
    
    # Read item
    response = table.get_item(Key={"user_id": "u-123"})
    print("Read:", response.get("Item"))

    # Cleanup
    table.delete()
    print("Dropped table:", TABLE_NAME)

if __name__ == "__main__":
    main()
```

## Extracting Data from DynamoDB Responses

Excellent work on learning how to create DynamoDB tables and handle timing issues! Now, let's focus on properly reading data from your table.

You have code that creates a table, adds a user item, and attempts to read it back. However, there's an issue with how the retrieved data is displayed. When you run the current code, instead of seeing clean user data, you'll see a messy response object filled with DynamoDB metadata.

Your task is to fix the print statement so that it properly extracts and displays just the user item data. Look for the TODO comment that shows you exactly where to make the change.

This will teach you how to handle DynamoDB read responses correctly and get clean, readable output in your applications.

```python
import time, uuid, boto3
from botocore.exceptions import ClientError

dynamodb = boto3.resource('dynamodb')
client = boto3.client('dynamodb')
TABLE_NAME = f"Users_{uuid.uuid4().hex[:8]}"

def wait_active(name):
    for _ in range(30):
        if client.describe_table(TableName=name)["Table"]["TableStatus"] == "ACTIVE":
            return
        time.sleep(2)
    raise TimeoutError("Table not active in time")

def main():
    # Create table
    table = dynamodb.create_table(
        TableName=TABLE_NAME,
        KeySchema=[{"AttributeName": "user_id", "KeyType": "HASH"}],
        AttributeDefinitions=[{"AttributeName": "user_id", "AttributeType": "S"}],
        BillingMode='PAY_PER_REQUEST'
    )
    wait_active(TABLE_NAME)
    print("Table created:", TABLE_NAME)

    # Create item
    table.put_item(Item={"user_id": "u-123", "name": "Ada", "email": "ada@example.com"})
    
    # Read item
    response = table.get_item(Key={"user_id": "u-123"})
    # TODO: Fix this print statement to extract the actual item data from the response
    print("Read:", response)

    # Cleanup
    table.delete()
    print("Dropped table:", TABLE_NAME)

if __name__ == "__main__":
    main()
```

Here is the completed code with the print statement fixed to extract the item data via `response.get("Item")`:

```python
import time, uuid, boto3
from botocore.exceptions import ClientError

dynamodb = boto3.resource('dynamodb')
client = boto3.client('dynamodb')
TABLE_NAME = f"Users_{uuid.uuid4().hex[:8]}"

def wait_active(name):
    for _ in range(30):
        if client.describe_table(TableName=name)["Table"]["TableStatus"] == "ACTIVE":
            return
        time.sleep(2)
    raise TimeoutError("Table not active in time")

def main():
    # Create table
    table = dynamodb.create_table(
        TableName=TABLE_NAME,
        KeySchema=[{"AttributeName": "user_id", "KeyType": "HASH"}],
        AttributeDefinitions=[{"AttributeName": "user_id", "AttributeType": "S"}],
        BillingMode='PAY_PER_REQUEST'
    )
    wait_active(TABLE_NAME)
    print("Table created:", TABLE_NAME)

    # Create item
    table.put_item(Item={"user_id": "u-123", "name": "Ada", "email": "ada@example.com"})
    
    # Read item
    response = table.get_item(Key={"user_id": "u-123"})
    print("Read:", response.get("Item"))

    # Cleanup
    table.delete()
    print("Dropped table:", TABLE_NAME)

if __name__ == "__main__":
    main()
```

## Fixing DynamoDB Update Expression Syntax

Perfect! You've mastered creating tables, handling timing issues, and reading data from DynamoDB. Now, let's focus on the Update operation, which is where many developers run into syntax challenges.

You have code that creates a table, adds a user, and attempts to update the user's email address. However, there's a syntax error in the update operation that will cause it to fail. When you run the current code, you'll see an error because the `UpdateExpression` is missing a required keyword.

Your task is to fix the `UpdateExpression` syntax in the `update_item` call. Look for the TODO comment that shows you exactly where to make the change. DynamoDB requires you to specify what type of operation you want to perform on the data.

Once you fix the syntax, your code will successfully update the user's email and display the updated item, completing your understanding of all four CRUD operations in DynamoDB.

```python
import time, uuid, boto3
from botocore.exceptions import ClientError

dynamodb = boto3.resource('dynamodb')
client = boto3.client('dynamodb')
TABLE_NAME = f"Users_{uuid.uuid4().hex[:8]}"

def wait_active(name):
    for _ in range(30):
        if client.describe_table(TableName=name)["Table"]["TableStatus"] == "ACTIVE":
            return
        time.sleep(2)
    raise TimeoutError("Table not active in time")

def main():
    # Create table
    table = dynamodb.create_table(
        TableName=TABLE_NAME,
        KeySchema=[{"AttributeName": "user_id", "KeyType": "HASH"}],
        AttributeDefinitions=[{"AttributeName": "user_id", "AttributeType": "S"}],
        BillingMode='PAY_PER_REQUEST'
    )
    wait_active(TABLE_NAME)
    print("Table created:", TABLE_NAME)

    # Create item
    table.put_item(Item={"user_id": "u-123", "name": "Ada", "email": "ada@example.com"})
    
    # Read item
    response = table.get_item(Key={"user_id": "u-123"})
    print("Read:", response.get("Item"))
    
    # Update item
    # TODO: Fix the UpdateExpression syntax - it needs to specify the type of operation
    updated = table.update_item(
        Key={"user_id": "u-123"},
        UpdateExpression="email = :e",
        ExpressionAttributeValues={":e": "ada.lovelace@example.com"},
        ReturnValues="ALL_NEW"
    )
    print("Updated:", updated["Attributes"])

    # Cleanup
    table.delete()
    print("Dropped table:", TABLE_NAME)

if __name__ == "__main__":
    main()
```

Here is the completed code with the `UpdateExpression` fixed by adding the required `SET` keyword:

```python
import time, uuid, boto3
from botocore.exceptions import ClientError

dynamodb = boto3.resource('dynamodb')
client = boto3.client('dynamodb')
TABLE_NAME = f"Users_{uuid.uuid4().hex[:8]}"

def wait_active(name):
    for _ in range(30):
        if client.describe_table(TableName=name)["Table"]["TableStatus"] == "ACTIVE":
            return
        time.sleep(2)
    raise TimeoutError("Table not active in time")

def main():
    # Create table
    table = dynamodb.create_table(
        TableName=TABLE_NAME,
        KeySchema=[{"AttributeName": "user_id", "KeyType": "HASH"}],
        AttributeDefinitions=[{"AttributeName": "user_id", "AttributeType": "S"}],
        BillingMode='PAY_PER_REQUEST'
    )
    wait_active(TABLE_NAME)
    print("Table created:", TABLE_NAME)

    # Create item
    table.put_item(Item={"user_id": "u-123", "name": "Ada", "email": "ada@example.com"})
    
    # Read item
    response = table.get_item(Key={"user_id": "u-123"})
    print("Read:", response.get("Item"))
    
    # Update item
    updated = table.update_item(
        Key={"user_id": "u-123"},
        UpdateExpression="SET email = :e",
        ExpressionAttributeValues={":e": "ada.lovelace@example.com"},
        ReturnValues="ALL_NEW"
    )
    print("Updated:", updated["Attributes"])

    # Cleanup
    table.delete()
    print("Dropped table:", TABLE_NAME)

if __name__ == "__main__":
    main()
```

## Completing DynamoDB Delete and Verification

Fantastic! You've successfully worked through table creation, data reading, and update operations. Now it's time to complete your CRUD knowledge by implementing the final operation: Delete.

You have code that creates a table, adds a user, reads the user data, and updates the user's email. However, the delete operation and its verification are missing from your workflow.

Your task is to complete two missing pieces:

* Add the `delete_item` call to remove the user from the table
* Add a verification read to confirm that the deletion worked

Look for the TODO comments that show you exactly where to add each piece. The first TODO asks you to delete the user with the correct `Key` parameter, and the second TODO asks you to verify the deletion by attempting to read the item again.

Once you complete both parts, you'll have implemented all four CRUD operations and learned the important practice of verifying that your database operations have succeeded!

```python
import time, uuid, boto3
from botocore.exceptions import ClientError

dynamodb = boto3.resource('dynamodb')
client = boto3.client('dynamodb')
TABLE_NAME = f"Users_{uuid.uuid4().hex[:8]}"

def wait_active(name):
    for _ in range(30):
        if client.describe_table(TableName=name)["Table"]["TableStatus"] == "ACTIVE":
            return
        time.sleep(2)
    raise TimeoutError("Table not active in time")

def main():
    # Create table
    table = dynamodb.create_table(
        TableName=TABLE_NAME,
        KeySchema=[{"AttributeName": "user_id", "KeyType": "HASH"}],
        AttributeDefinitions=[{"AttributeName": "user_id", "AttributeType": "S"}],
        BillingMode='PAY_PER_REQUEST'
    )
    wait_active(TABLE_NAME)
    print("Table created:", TABLE_NAME)

    # Create item
    table.put_item(Item={"user_id": "u-123", "name": "Ada", "email": "ada@example.com"})
    
    # Read item
    response = table.get_item(Key={"user_id": "u-123"})
    print("Read:", response.get("Item"))
    
    # Update item
    updated = table.update_item(
        Key={"user_id": "u-123"},
        UpdateExpression="SET email = :e",
        ExpressionAttributeValues={":e": "ada.lovelace@example.com"},
        ReturnValues="ALL_NEW"
    )
    print("Updated:", updated["Attributes"])
    
    # Delete item
    # TODO: Add the delete_item call to remove the user with user_id "u-123"
    print("Deleted user u-123")
    
    # Verify deletion
    # TODO: Add a get_item call to verify the user was deleted and print the result
    
    # Cleanup
    table.delete()
    print("Dropped table:", TABLE_NAME)

if __name__ == "__main__":
    main()
```

Here is the completed code with the `delete_item` call and the deletion verification added:

```python
import time, uuid, boto3
from botocore.exceptions import ClientError

dynamodb = boto3.resource('dynamodb')
client = boto3.client('dynamodb')
TABLE_NAME = f"Users_{uuid.uuid4().hex[:8]}"

def wait_active(name):
    for _ in range(30):
        if client.describe_table(TableName=name)["Table"]["TableStatus"] == "ACTIVE":
            return
        time.sleep(2)
    raise TimeoutError("Table not active in time")

def main():
    # Create table
    table = dynamodb.create_table(
        TableName=TABLE_NAME,
        KeySchema=[{"AttributeName": "user_id", "KeyType": "HASH"}],
        AttributeDefinitions=[{"AttributeName": "user_id", "AttributeType": "S"}],
        BillingMode='PAY_PER_REQUEST'
    )
    wait_active(TABLE_NAME)
    print("Table created:", TABLE_NAME)

    # Create item
    table.put_item(Item={"user_id": "u-123", "name": "Ada", "email": "ada@example.com"})
    
    # Read item
    response = table.get_item(Key={"user_id": "u-123"})
    print("Read:", response.get("Item"))
    
    # Update item
    updated = table.update_item(
        Key={"user_id": "u-123"},
        UpdateExpression="SET email = :e",
        ExpressionAttributeValues={":e": "ada.lovelace@example.com"},
        ReturnValues="ALL_NEW"
    )
    print("Updated:", updated["Attributes"])
    
    # Delete item
    table.delete_item(Key={"user_id": "u-123"})
    print("Deleted user u-123")
    
    # Verify deletion
    verify = table.get_item(Key={"user_id": "u-123"})
    print("Verify deletion:", verify.get("Item"))
    
    # Cleanup
    table.delete()
    print("Dropped table:", TABLE_NAME)

if __name__ == "__main__":
    main()
```

## Adding DynamoDB Resource Cleanup

Wonderful! You've now mastered all four CRUD operations and learned how to verify your database changes. There's one final piece that completes the responsible AWS developer workflow: proper resource cleanup.

You have code that demonstrates a complete CRUD workflow — it creates a DynamoDB table, performs all four operations on user data, and everything works perfectly. However, there's a crucial step missing at the end that every AWS developer must include.

When you create AWS resources like DynamoDB tables, they continue to exist (and potentially cost money) until you explicitly remove them. Your task is to add the missing cleanup step that deletes the table after all operations are complete.

Look for the TODO comment at the end of the `main` function, which shows you exactly where to add the table deletion. You'll need to add both the deletion call and a print statement to confirm the cleanup happened.

This simple addition will complete your understanding of the full AWS resource lifecycle and teach you an essential best practice for all your future cloud development work!

```python
import time, uuid, boto3
from botocore.exceptions import ClientError

dynamodb = boto3.resource('dynamodb')
client = boto3.client('dynamodb')
TABLE_NAME = f"Users_{uuid.uuid4().hex[:8]}"

def wait_active(name):
    for _ in range(30):
        if client.describe_table(TableName=name)["Table"]["TableStatus"] == "ACTIVE":
            return
        time.sleep(2)
    raise TimeoutError("Table not active in time")

def main():
    # Create table
    table = dynamodb.create_table(
        TableName=TABLE_NAME,
        KeySchema=[{"AttributeName": "user_id", "KeyType": "HASH"}],
        AttributeDefinitions=[{"AttributeName": "user_id", "AttributeType": "S"}],
        BillingMode='PAY_PER_REQUEST'
    )
    wait_active(TABLE_NAME)
    print("Table created:", TABLE_NAME)

    # Create item
    table.put_item(Item={"user_id": "u-123", "name": "Ada", "email": "ada@example.com"})
    
    # Read item
    response = table.get_item(Key={"user_id": "u-123"})
    print("Read:", response.get("Item"))
    
    # Update item
    updated = table.update_item(
        Key={"user_id": "u-123"},
        UpdateExpression="SET email = :e",
        ExpressionAttributeValues={":e": "ada.lovelace@example.com"},
        ReturnValues="ALL_NEW"
    )
    print("Updated:", updated["Attributes"])
    
    # Delete item
    table.delete_item(Key={"user_id": "u-123"})
    print("Deleted user u-123")
    
    # TODO: Add table cleanup - delete the table and print a confirmation message

if __name__ == "__main__":
    main()
```

Here is the completed code with the table cleanup added at the end of `main`:

```python
import time, uuid, boto3
from botocore.exceptions import ClientError

dynamodb = boto3.resource('dynamodb')
client = boto3.client('dynamodb')
TABLE_NAME = f"Users_{uuid.uuid4().hex[:8]}"

def wait_active(name):
    for _ in range(30):
        if client.describe_table(TableName=name)["Table"]["TableStatus"] == "ACTIVE":
            return
        time.sleep(2)
    raise TimeoutError("Table not active in time")

def main():
    # Create table
    table = dynamodb.create_table(
        TableName=TABLE_NAME,
        KeySchema=[{"AttributeName": "user_id", "KeyType": "HASH"}],
        AttributeDefinitions=[{"AttributeName": "user_id", "AttributeType": "S"}],
        BillingMode='PAY_PER_REQUEST'
    )
    wait_active(TABLE_NAME)
    print("Table created:", TABLE_NAME)

    # Create item
    table.put_item(Item={"user_id": "u-123", "name": "Ada", "email": "ada@example.com"})
    
    # Read item
    response = table.get_item(Key={"user_id": "u-123"})
    print("Read:", response.get("Item"))
    
    # Update item
    updated = table.update_item(
        Key={"user_id": "u-123"},
        UpdateExpression="SET email = :e",
        ExpressionAttributeValues={":e": "ada.lovelace@example.com"},
        ReturnValues="ALL_NEW"
    )
    print("Updated:", updated["Attributes"])
    
    # Delete item
    table.delete_item(Key={"user_id": "u-123"})
    print("Deleted user u-123")
    
    # Cleanup
    table.delete()
    print("Dropped table:", TABLE_NAME)

if __name__ == "__main__":
    main()
```